# EKSPERIMEN LANJUTAN: TF-IDF + Keyword Density Features
---
**Ekstensi dari:** `TEST03_2_improved.ipynb`

> **Strategi:** Menambahkan fitur diskriminatif berbasis keyword khas per genre
> di atas TF-IDF yang sudah ada, kemudian menggabungkannya dengan `scipy.sparse.hstack`.
>
> **Prasyarat:** Jalankan seluruh notebook `TEST03_2_improved.ipynb` terlebih dahulu
> sehingga variabel `X_train`, `X_test`, `X_train_text`, `X_test_text`,
> `y_train`, `y_test`, `df_balanced`, `evaluate_model`, dan `manual_confusion_matrix`
> sudah tersedia di memori Colab.

Pipeline penggabungan fitur:
```
X_train_text  ──► TF-IDF ──────────────────────┐
                                                 ├─► hstack ──► LinearSVC
X_train_text  ──► Keyword Density (17 fitur) ──┘
```

## Import tambahan

In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

print("Import selesai.")

## Definisi Keyword Dictionary per Genre

Keyword dipilih berdasarkan ciri khas tiap genre:
- **Action**: kata yang berhubungan dengan combat, senjata, pertempuran langsung
- **RPG**: kata yang berhubungan dengan progression, karakter, quest, magic
- **Simulation**: kata yang berhubungan dengan manajemen, pembangunan, ekonomi
- **Adventure**: kata yang berhubungan dengan eksplorasi, narasi, puzzle, cerita

> Keyword sengaja **tidak di-stemming** agar lebih mudah diinterpretasi dan
> bisa ditambah/diubah secara manual sesuai analisis data Anda.

In [ ]:
# ── Keyword Dictionary per Genre ─────────────────────────────────────────────
# Setiap kata dipilih karena diskriminatif (jarang muncul di genre lain)

GENRE_KEYWORDS = {
    'action': [
        'shoot', 'shooter', 'gun', 'guns', 'weapon', 'weapons',
        'combat', 'fight', 'fighting', 'battle', 'battles',
        'enemy', 'enemies', 'kill', 'killing', 'attack', 'attacks',
        'fps', 'brawl', 'brawler', 'arena', 'deathmatch',
        'warfare', 'tactical', 'stealth', 'assassin',
    ],
    'rpg': [
        'rpg', 'role', 'quest', 'quests', 'level', 'leveling',
        'skill', 'skills', 'stat', 'stats', 'magic', 'spell', 'spells',
        'character', 'characters', 'class', 'classes',
        'inventory', 'loot', 'dungeon', 'dungeons',
        'experience', 'xp', 'upgrade', 'upgrades',
        'hero', 'heroes', 'fantasy', 'wizard', 'mage',
    ],
    'simulation': [
        'simulat', 'manage', 'management', 'build', 'building',
        'farm', 'farming', 'economy', 'economic',
        'city', 'town', 'construct', 'resource', 'resources',
        'strategy', 'tycoon', 'sandbox', 'craft', 'crafting',
        'factory', 'colony', 'base', 'engineer',
    ],
    'adventure': [
        'adventur', 'explor', 'exploration', 'puzzle', 'puzzles',
        'story', 'narrative', 'mystery', 'discover', 'journey',
        'world', 'open world', 'quest', 'treasure',
        'platform', 'platformer', 'point and click',
        'survival', 'horror', 'escape', 'hidden',
    ],
}

# Verifikasi
for genre, kws in GENRE_KEYWORDS.items():
    print(f"{genre:12s}: {len(kws):2d} keywords  |  {', '.join(kws[:5])}...")

## Fungsi Ekstraksi Keyword Density Features

Setiap teks menghasilkan **17 fitur numerik**:
- 4 fitur: jumlah keyword yang cocok per genre (raw count)
- 4 fitur: density (count / panjang teks) per genre
- 4 fitur: flag biner (1 jika ada keyword genre tsb, 0 jika tidak)
- 1 fitur: panjang sinopsis (word count)
- 4 fitur: rasio antar genre (action/rpg, sim/adv, dll)

> Fitur density lebih robust dari raw count karena tidak terpengaruh panjang teks.

In [ ]:
def extract_keyword_features(texts, genre_kw=GENRE_KEYWORDS):
    '''
    Ekstrak 17 fitur keyword density dari list teks.
    Input : list of str
    Output: scipy sparse matrix shape (n_samples, 17)
    '''
    genre_names = ['action', 'rpg', 'simulation', 'adventure']
    rows = []

    for text in texts:
        t = str(text).lower()
        words = t.split()
        n_words = max(len(words), 1)

        # Hitung match per genre
        counts = {}
        for g in genre_names:
            cnt = sum(1 for kw in genre_kw[g] if kw in t)
            counts[g] = cnt

        # 4 fitur: raw count
        raw = [counts[g] for g in genre_names]

        # 4 fitur: density (count / panjang)
        density = [counts[g] / n_words for g in genre_names]

        # 4 fitur: flag biner (ada keyword atau tidak)
        flag = [1 if counts[g] > 0 else 0 for g in genre_names]

        # 1 fitur: panjang sinopsis (ternormalisasi log)
        import math
        length = math.log1p(n_words)

        # 4 fitur: rasio keyword antar genre
        total_kw = sum(counts.values()) + 1e-9
        ratio = [counts[g] / total_kw for g in genre_names]

        row = raw + density + flag + [length] + ratio
        rows.append(row)

    arr = sp.csr_matrix(rows)
    return arr

# Test cepat
test_text = ["Shoot enemies with guns and fight in arena battles",
             "Level up your character, learn spells and complete quests",
             "Build a farm, manage resources and grow your economy",
             "Explore mysterious dungeons and solve puzzles in this adventure"]

sample = extract_keyword_features(test_text)
print(f"Shape fitur sample: {sample.shape}")
print(f"Fitur teks 1 (action) : {sample[0].toarray().round(3)}")
print(f"Fitur teks 2 (rpg)    : {sample[1].toarray().round(3)}")
print(f"Fitur teks 3 (sim)    : {sample[2].toarray().round(3)}")
print(f"Fitur teks 4 (adv)    : {sample[3].toarray().round(3)}")

## Analisis Keyword: Seberapa Diskriminatif?

Cek frekuensi kemunculan keyword di masing-masing genre pada data latih.
Ini membantu memvalidasi apakah keyword yang dipilih benar-benar membedakan genre.

In [ ]:
# Gabungkan text dengan label untuk analisis
df_analysis = pd.DataFrame({
    'text' : X_train_text.values,
    'genre': y_train.values
})

genre_names = ['action', 'rpg', 'simulation', 'adventure']
analysis_rows = []

for genre in genre_names:
    subset = df_analysis[df_analysis['genre'] == genre]['text']
    total  = len(subset)
    for kw in GENRE_KEYWORDS[genre]:
        hits = subset.apply(lambda t: kw in str(t).lower()).sum()
        analysis_rows.append({
            'Genre'       : genre,
            'Keyword'     : kw,
            'Muncul'      : hits,
            'Total data'  : total,
            'Coverage (%)'  : round(hits / total * 100, 1),
        })

df_kw_analysis = pd.DataFrame(analysis_rows)

# Tampilkan top-5 keyword per genre berdasarkan coverage
print("=== Top-5 Keyword Paling Sering Muncul per Genre (data latih) ===\n")
for genre in genre_names:
    top = (df_kw_analysis[df_kw_analysis['Genre'] == genre]
           .sort_values('Coverage (%)', ascending=False)
           .head(5))
    print(f"► {genre.upper()}")
    print(top[['Keyword','Muncul','Coverage (%)']].to_string(index=False))
    print()

In [ ]:
# Visualisasi: heatmap coverage keyword per genre
import matplotlib.pyplot as plt
import seaborn as sns

pivot = df_kw_analysis.pivot_table(
    index='Keyword', columns='Genre', values='Coverage (%)', fill_value=0)

# Ambil keyword yang coverage-nya > 5% di minimal satu genre
pivot_filtered = pivot[pivot.max(axis=1) > 5]

plt.figure(figsize=(10, max(6, len(pivot_filtered) * 0.35)))
sns.heatmap(pivot_filtered, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Coverage (%)'})
plt.title('Keyword Coverage per Genre (data latih, %)')
plt.xlabel('Genre')
plt.ylabel('Keyword')
plt.tight_layout()
plt.show()
print(f"Keyword ditampilkan: {len(pivot_filtered)} dari {len(pivot)} total")

## Penggabungan Fitur: TF-IDF + Keyword Density

Menggunakan `scipy.sparse.hstack` untuk menggabungkan dua matriks sparse:
- `X_train` (TF-IDF, shape ~[n, 20000])
- `kw_train` (keyword features, shape ~[n, 17])

Fitur keyword di-scale dengan `MaxAbsScaler` agar skalanya tidak jauh berbeda
dari nilai TF-IDF (yang umumnya kecil, 0–1).

In [ ]:
# Ekstrak keyword features
print("Mengekstrak keyword features...")
kw_train_raw = extract_keyword_features(X_train_text.tolist())
kw_test_raw  = extract_keyword_features(X_test_text.tolist())

print(f"Shape keyword features latih : {kw_train_raw.shape}")
print(f"Shape keyword features uji   : {kw_test_raw.shape}")

# Scale agar rentang nilai sesuai dengan TF-IDF
scaler = MaxAbsScaler()
kw_train = scaler.fit_transform(kw_train_raw)   # fit pada train saja
kw_test  = scaler.transform(kw_test_raw)

# Gabungkan TF-IDF + Keyword
X_train_combined = sp.hstack([X_train, kw_train])
X_test_combined  = sp.hstack([X_test,  kw_test])

print(f"\nShape matriks gabungan latih : {X_train_combined.shape}")
print(f"Shape matriks gabungan uji   : {X_test_combined.shape}")
print(f"Jumlah fitur TF-IDF          : {X_train.shape[1]}")
print(f"Jumlah fitur keyword         : {kw_train.shape[1]}")
print(f"Total fitur gabungan         : {X_train_combined.shape[1]}")

## GridSearchCV: Cari Parameter C Optimal untuk LinearSVC

Parameter `C` (regularization) sangat berpengaruh pada LinearSVC.
Nilai kecil = regularization kuat (model lebih general), nilai besar = fit lebih ketat.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

print("Mencari nilai C optimal dengan GridSearchCV (5-fold)...")
print("Ini bisa memakan waktu 1-3 menit...\n")

param_grid = {'C': [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]}
skf_gs = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs_svm = GridSearchCV(
    LinearSVC(class_weight='balanced', max_iter=10000, random_state=42),
    param_grid=param_grid,
    cv=skf_gs,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
gs_svm.fit(X_train_combined, y_train)

best_C = gs_svm.best_params_['C']
print(f"\nC terbaik     : {best_C}")
print(f"F1 CV terbaik : {gs_svm.best_score_ * 100:.2f}%")

# Tabel hasil GridSearch
gs_df = pd.DataFrame({
    'C'       : param_grid['C'],
    'F1 CV'   : [f"{s*100:.2f}%" for s in gs_svm.cv_results_['mean_test_score']],
    'Std'     : [f"+/- {s*100:.2f}%" for s in gs_svm.cv_results_['std_test_score']],
})
display(gs_df)

## Latih Model dengan Fitur Gabungan (TF-IDF + Keyword Density)

Tiga model dilatih pada `X_combined`:
1. **LinearSVC** dengan C optimal (model utama)
2. **LinearSVC** baseline (C=1.0, untuk perbandingan)
3. **KNN cosine** dengan k dari GridSearchCV sebelumnya (jika tersedia)

> Naive Bayes tidak diikutkan karena tidak mendukung nilai fitur negatif
> yang bisa muncul setelah scaling.

In [ ]:
# Model 1: LinearSVC dengan C optimal (model utama perbaikan)
svm_combined = LinearSVC(
    C=best_C, class_weight='balanced', max_iter=10000, random_state=42)
svm_combined.fit(X_train_combined, y_train)
y_pred_svm_combined = svm_combined.predict(X_test_combined)
print("LinearSVC (C optimal, fitur gabungan) — selesai")

# Model 2: LinearSVC baseline C=1.0 (untuk perbandingan adil)
svm_combined_base = LinearSVC(
    C=1.0, class_weight='balanced', max_iter=10000, random_state=42)
svm_combined_base.fit(X_train_combined, y_train)
y_pred_svm_base = svm_combined_base.predict(X_test_combined)
print("LinearSVC (C=1.0, fitur gabungan)     — selesai")

# Model 3: LinearSVC hanya TF-IDF (kontrol — sudah ada dari notebook sebelumnya)
# Pakai y_pred_svm yang sudah dihitung sebelumnya sebagai baseline
print("\nSemua model selesai dilatih.")

## Evaluasi Model: Perbandingan Sebelum vs Sesudah Keyword Features

Evaluasi lengkap: confusion matrix, tabel manual TP/FP/FN/TN, dan ringkasan metrik.

In [ ]:
print("=" * 60)
print("EVALUASI: LinearSVC + TF-IDF + Keyword Density (C optimal)")
print("=" * 60)
f1_combined_opt = evaluate_model(
    y_test, y_pred_svm_combined,
    f'LinearSVC+KW (C={best_C}, fitur gabungan)')

In [ ]:
print("=" * 60)
print("EVALUASI: LinearSVC + TF-IDF + Keyword Density (C=1.0)")
print("=" * 60)
f1_combined_base = evaluate_model(
    y_test, y_pred_svm_base,
    'LinearSVC+KW (C=1.0, fitur gabungan)')

## Tabel Ringkasan: Perbandingan Semua Pendekatan

Membandingkan semua model dari skenario TF-IDF saja (dari notebook sebelumnya)
dengan model TF-IDF + Keyword Density.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def get_metrics(y_true, y_pred):
    return {
        'Accuracy' : f"{accuracy_score(y_true, y_pred)*100:.2f}%",
        'Precision': f"{precision_score(y_true, y_pred, average='macro', zero_division=0)*100:.2f}%",
        'Recall'   : f"{recall_score(y_true, y_pred, average='macro', zero_division=0)*100:.2f}%",
        'F1'       : f"{f1_score(y_true, y_pred, average='macro', zero_division=0)*100:.2f}%",
    }

# Kumpulkan semua hasil
# Pastikan y_pred_svm tersedia dari notebook sebelumnya (LinearSVC TF-IDF saja)
comparison_rows = []

try:
    comparison_rows.append({'Model': 'LinearSVM (TF-IDF saja, S1)',
                             'Fitur': 'TF-IDF only', **get_metrics(y_test, y_pred_svm)})
except NameError:
    print("Catatan: y_pred_svm tidak ditemukan, skip baseline TF-IDF-only")

comparison_rows.append({
    'Model': f'LinearSVM+KW (C={best_C})',
    'Fitur': 'TF-IDF + Keyword', **get_metrics(y_test, y_pred_svm_combined)})

comparison_rows.append({
    'Model': 'LinearSVM+KW (C=1.0)',
    'Fitur': 'TF-IDF + Keyword', **get_metrics(y_test, y_pred_svm_base)})

df_comparison = pd.DataFrame(comparison_rows)
print("=== Ringkasan Perbandingan ===")
display(df_comparison)

# Hitung selisih F1
baseline_f1 = 77.27  # F1 terbaik dari notebook sebelumnya (LinearSVM S1)
best_new_f1 = max(
    f1_score(y_test, y_pred_svm_combined, average='macro', zero_division=0),
    f1_score(y_test, y_pred_svm_base,     average='macro', zero_division=0),
) * 100

print(f"\nBaseline F1 (TF-IDF saja) : {baseline_f1:.2f}%")
print(f"F1 terbaik (TF-IDF + KW)  : {best_new_f1:.2f}%")
print(f"Selisih peningkatan       : +{best_new_f1 - baseline_f1:.2f}%")

## Analisis: Fitur Keyword Mana yang Paling Berpengaruh?

`LinearSVC` menyimpan koefisien per fitur per kelas — kita bisa melihat
keyword mana yang paling berkontribusi dalam membedakan genre.

In [ ]:
# Ambil nama semua fitur
tfidf_feature_names = tfidf.get_feature_names_out().tolist()

genre_names_ordered = ['action', 'rpg', 'simulation', 'adventure']
kw_feature_names = []
for g in genre_names_ordered:
    kw_feature_names += [f'KW_count_{g}']
for g in genre_names_ordered:
    kw_feature_names += [f'KW_density_{g}']
for g in genre_names_ordered:
    kw_feature_names += [f'KW_flag_{g}']
kw_feature_names += ['KW_length']
for g in genre_names_ordered:
    kw_feature_names += [f'KW_ratio_{g}']

all_feature_names = tfidf_feature_names + kw_feature_names

print(f"Total fitur: {len(all_feature_names)}")
print(f"  TF-IDF  : {len(tfidf_feature_names)}")
print(f"  Keyword : {len(kw_feature_names)}")

# Koefisien model LinearSVC (one-vs-rest)
classes = svm_combined.classes_
coef    = svm_combined.coef_   # shape: (n_classes, n_features)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (cls, ax) in enumerate(zip(classes, axes)):
    coef_row = coef[idx]
    # Ambil hanya fitur keyword (bagian akhir)
    kw_coef = coef_row[-len(kw_feature_names):]
    kw_df   = pd.DataFrame({'Fitur': kw_feature_names, 'Koefisien': kw_coef})
    kw_df   = kw_df.sort_values('Koefisien', ascending=False)

    # Plot top 10 positif dan top 5 negatif
    top_pos = kw_df.head(8)
    top_neg = kw_df.tail(5)
    plot_df = pd.concat([top_pos, top_neg])

    colors = ['#185FA5' if v > 0 else '#A32D2D' for v in plot_df['Koefisien']]
    ax.barh(plot_df['Fitur'], plot_df['Koefisien'], color=colors)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(f'Koefisien keyword — kelas: {cls.upper()}', fontsize=12)
    ax.set_xlabel('Koefisien SVM')
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('Kontribusi Fitur Keyword per Genre (LinearSVC)', fontsize=13)
plt.tight_layout()
plt.show()

## Validasi Silang K-Fold (5-Fold) dengan Fitur Gabungan

Menggunakan custom transformer agar keyword features bisa masuk ke Pipeline sklearn.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import FeatureUnion, Pipeline

# Custom transformer untuk keyword features (agar bisa masuk Pipeline)
class KeywordFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, genre_kw=None):
        self.genre_kw = genre_kw or GENRE_KEYWORDS

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return extract_keyword_features(list(X))

# Pipeline TF-IDF + Keyword dengan FeatureUnion
tfidf_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=2, max_df=0.95)),
])

combined_features = FeatureUnion([
    ('tfidf',    TfidfVectorizer(
        max_features=20000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=2, max_df=0.95)),
    ('keywords', KeywordFeatureExtractor()),
])

full_pipeline = Pipeline([
    ('features', combined_features),
    ('scaler',   MaxAbsScaler()),
    ('clf',      LinearSVC(C=best_C, class_weight='balanced',
                           max_iter=10000, random_state=42)),
])

# K-Fold
X_cv = df_balanced['synopsis_S3']
y_cv = df_balanced['genre_main']
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

from sklearn.model_selection import cross_validate
scoring = {
    'Accuracy' : 'accuracy',
    'Precision': 'precision_macro',
    'Recall'   : 'recall_macro',
    'F1-Score' : 'f1_macro',
}

print("Menjalankan 5-fold cross-validation (TF-IDF + Keyword Features)...")
cv_scores = cross_validate(full_pipeline, X_cv, y_cv, cv=skf,
                           scoring=scoring, n_jobs=-1)

print("\n=== Hasil K-Fold (TF-IDF + Keyword Density) ===")
for metric_name, key in [('Accuracy','test_Accuracy'), ('Precision','test_Precision'),
                          ('Recall','test_Recall'),    ('F1-Score','test_F1-Score')]:
    mean = cv_scores[key].mean() * 100
    std  = cv_scores[key].std()  * 100
    print(f"  {metric_name:10s}: {mean:.2f}% +/- {std:.2f}%")

## Tabel MSE & RMSE

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import numpy as np

le = LabelEncoder().fit(df_balanced['genre_main'])

mse_rows = []
for name, yp in [
    (f'LinearSVM+KW C={best_C}', y_pred_svm_combined),
    ('LinearSVM+KW C=1.0',       y_pred_svm_base),
]:
    mse  = mean_squared_error(le.transform(y_test), le.transform(yp))
    rmse = np.sqrt(mse)
    mse_rows.append({'Model': name, 'MSE': round(mse,4), 'RMSE': round(rmse,4)})

pd.DataFrame(mse_rows).sort_values('MSE').reset_index(drop=True)

## Implementasi: Inferensi Genre dengan Model Gabungan

Fungsi `predict_genre_combined` menggunakan pipeline lengkap
(TF-IDF + Keyword Density + LinearSVC) untuk memprediksi genre sinopsis baru.

In [ ]:
# Latih ulang full_pipeline pada SELURUH data balanced untuk inferensi
print("Melatih pipeline final pada seluruh data balanced...")
full_pipeline.fit(df_balanced['synopsis_S3'], df_balanced['genre_main'])
print("Pipeline final siap.\n")

def predict_genre_combined(synopsis_text):
    '''
    Prediksi genre dari teks sinopsis menggunakan model gabungan
    TF-IDF + Keyword Density Features + LinearSVC.
    '''
    # Preprocessing S3
    processed = preprocess(synopsis_text, use_stopword=True, use_stemming=True)
    pred = full_pipeline.predict([processed])[0]
    return pred

# ── Uji dengan contoh sinopsis ──────────────────────────────────────────────
contoh_list = [
    ("A fast-paced shooter where you eliminate enemies using assault rifles and grenades in intense arena combat.",
     "action"),
    ("Embark on an epic quest, level up your wizard character, learn powerful spells and defeat the dragon.",
     "rpg"),
    ("Build and manage your own city, balance economy and resources to keep citizens happy in this tycoon simulator.",
     "simulation"),
    ("Explore a mysterious island, solve ancient puzzles and uncover the narrative secrets of a lost civilization.",
     "adventure"),
    ("An open-world fantasy RPG where you explore dungeons, level up your hero, and battle epic bosses.",
     "rpg/adventure"),
]

print("=== Hasil Prediksi Model TF-IDF + Keyword Density ===\n")
for synopsis, expected in contoh_list:
    pred = predict_genre_combined(synopsis)
    status = "✓" if expected.startswith(pred) else "~"
    print(f"{status} Prediksi : {pred.upper():<12} | Ekspektasi: {expected}")
    print(f"  Sinopsis : {synopsis[:75]}...")
    print()

## Kesimpulan Eksperimen Keyword Density Features

| Aspek | Nilai |
|---|---|
| Baseline F1 (TF-IDF saja, LinearSVM S1) | 77.27% |
| F1 dengan TF-IDF + Keyword Density | *lihat hasil di atas* |
| Jumlah fitur tambahan | 17 fitur |
| Teknik penggabungan | `scipy.sparse.hstack` |
| Scaling | `MaxAbsScaler` |

**Interpretasi:**
- Jika peningkatan < 1%: genre memang susah dibedakan dari kata saja; pertimbangkan Sentence-BERT
- Jika peningkatan 1–3%: keyword features cukup membantu sebagai fitur tambahan
- Jika peningkatan > 3%: keyword features sangat diskriminatif untuk dataset ini

**Untuk thesis:** eksperimen ini bisa dilaporkan sebagai *feature engineering* tambahan
yang menggabungkan pendekatan statistik (TF-IDF) dengan pendekatan berbasis pengetahuan
domain (*domain knowledge keyword dictionary*).